# 03 — ABC–XYZ segmentation and optimization-scope audit

The backend pipeline is the canonical classifier. Run `python src/backend/main.py` from the repository root to regenerate `sku_class.csv`, `sku_metric.csv`, and `optimization_scope.csv`. This notebook is read-only: it explains configured rules and audits the resulting artifacts without reclassifying SKUs.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src" / "backend").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "backend").exists():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")

PATHS = {
    "config": ROOT / "src" / "backend" / "config" / "config.json",
    "sku_class": ROOT / "data" / "processed" / "sku_class.csv",
    "sku_metric": ROOT / "data" / "processed" / "sku_metric.csv",
    "optimization_scope": ROOT / "data" / "processed" / "optimization_scope.csv",
}
missing = [str(path.relative_to(ROOT)) for path in PATHS.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing canonical segmentation artifacts. Run `python src/backend/main.py`: "
        + ", ".join(missing)
    )

config = json.loads(PATHS["config"].read_text(encoding="utf-8"))
sku_class = pd.read_csv(PATHS["sku_class"])
sku_metric = pd.read_csv(PATHS["sku_metric"])
optimization_scope = pd.read_csv(PATHS["optimization_scope"])

{
    "classified_skus": sku_class["sku_id"].nunique(),
    "metric_skus": sku_metric["sku_id"].nunique(),
    "selected_classes": optimization_scope["class"].tolist(),
}

## Gross-margin ABC

ABC prioritizes pre-holdout **latent demand value**, not fulfilled sales. For each row, economic value is `demand × max(unit_price − unit_cost, 0)`. This prevents a stocked-out SKU from being demoted merely because constrained sales were low and aligns the ranking with the lost-unit gross-margin cost used in simulation.

SKUs are sorted by total demand value. Cumulative shares through the configured A threshold are labeled A, shares through the B threshold are labeled B, and the remainder are C.

In [ ]:
abc_xyz_config = config["abc_xyz"]
backtest_config = config["backtest"]
data_config = config["data"]
scope_config = config["optimization_scope"]

configured_rules = pd.DataFrame(
    [
        {"rule": "ABC economic-value basis", "value": abc_xyz_config["abc_value_basis"]},
        {"rule": "ABC A cumulative threshold", "value": abc_xyz_config["abc_a_threshold"]},
        {"rule": "ABC B cumulative threshold", "value": abc_xyz_config["abc_b_threshold"]},
        {"rule": "XYZ lower relative quantile", "value": abc_xyz_config["xyz_q1"]},
        {"rule": "XYZ upper relative quantile", "value": abc_xyz_config["xyz_q3"]},
        {"rule": "XYZ seasonal lag days", "value": abc_xyz_config["xyz_season_length"]},
        {"rule": "XYZ rolling validation folds", "value": backtest_config["validation_n_folds"]},
        {"rule": "Days per XYZ validation fold", "value": backtest_config["validation_fold_days"]},
        {"rule": "Locked final holdout begins", "value": data_config["final_test_start"]},
    ]
)
configured_rules

## Causal, portfolio-relative XYZ

XYZ is a forecastability ranking, not a demand-CV label and not an absolute claim that a SKU is easy or hard to forecast. The backend compares causal rolling-one-step Naive, Seasonal Naive, and Historic Mean candidates on the same latest pre-holdout validation dates. A prediction for day `t` uses demand observed only through `t-1`; the locked final holdout is excluded.

The validation-selected candidate produces per-SKU normalized MAE. SKUs below the configured lower quantile are X, those through the upper quantile are Y, and the remainder are Z.

In [ ]:
class_distribution = (
    sku_class.groupby("class", observed=True)
    .agg(sku_count=("sku_id", "nunique"))
    .reset_index()
    .sort_values("class")
)
classification_audit = pd.DataFrame(
    [
        {"check": "one class row per SKU", "passed": not sku_class.duplicated("sku_id").any()},
        {"check": "all class codes match ABC × XYZ", "passed": bool(sku_class["class"].str.fullmatch(r"[ABC][XYZ]").all())},
        {"check": "one metric row per SKU", "passed": not sku_metric.duplicated("sku_id").any()},
        {"check": "class and metric SKU populations match", "passed": set(sku_class["sku_id"]) == set(sku_metric["sku_id"])},
        {"check": "all fill rates are probabilities", "passed": bool(sku_metric["fill_rate"].between(0, 1).all())},
        {"check": "lost sales are nonnegative", "passed": bool(sku_metric["lost_sales"].ge(0).all())},
    ]
)
display(classification_audit)
display(class_distribution)

## Dynamic optimization scope

The intervention scope is selected from pre-holdout business conditions rather than a hard-coded list of class names:

- `protect_strategic_value`: A classes whose share of A-class gross-margin demand value meets the configured minimum.
- `correct_understock`: C classes whose fill rate is below the configured threshold.
- `reduce_overstock`: classes with high fill rate and DOI at or above the configured portfolio quantile.

Because these rules are data-driven, selected classes may change when the source portfolio changes.

In [ ]:
scope_rules = pd.DataFrame(
    [
        {"rule": "minimum A-value share", "value": scope_config["high_value_a_share_min"]},
        {"rule": "understock fill-rate maximum", "value": scope_config["understock_fill_rate_max"]},
        {"rule": "overstock fill-rate minimum", "value": scope_config["overstock_fill_rate_min"]},
        {"rule": "overstock DOI quantile", "value": scope_config["overstock_doi_quantile"]},
    ]
)
scope_audit = pd.DataFrame(
    [
        {"check": "one decision per selected class", "passed": not optimization_scope.duplicated("class").any()},
        {"check": "selected classes exist in sku_class", "passed": set(optimization_scope["class"]).issubset(set(sku_class["class"]))},
        {"check": "scope SKU counts are positive", "passed": bool(optimization_scope["sku_count"].gt(0).all())},
        {"check": "scope fill rates are probabilities", "passed": bool(optimization_scope["fill_rate"].between(0, 1).all())},
        {"check": "scope DOI is nonnegative", "passed": bool(optimization_scope["DOI"].ge(0).all())},
    ]
)
scope_coverage = pd.DataFrame(
    [
        {
            "selected_classes": optimization_scope["class"].nunique(),
            "selected_skus": int(optimization_scope["sku_count"].sum()),
            "portfolio_skus": sku_class["sku_id"].nunique(),
            "sku_coverage": optimization_scope["sku_count"].sum() / sku_class["sku_id"].nunique(),
            "selected_pre_holdout_demand": float(optimization_scope["total_demand"].sum()),
            "portfolio_pre_holdout_demand": float(sku_metric["total_demand"].sum()),
            "demand_coverage": optimization_scope["total_demand"].sum() / sku_metric["total_demand"].sum(),
        }
    ]
)
display(scope_rules)
display(scope_audit)
display(scope_coverage)
display(optimization_scope.sort_values(["intervention", "class"]))